# Step 3.5. Further Preprocessing

이 노트북은 팀원이 Step 3까지 전처리한 `BankChurners_Final_Preprocessed.csv`를 기반으로 Step 4 Clustering과 Step 5 Segment Profiling에 사용할 최종 입력 데이터를 만든다.

핵심 원칙은 다음과 같다.

- Clustering은 고객 행동 feature만 사용한다.
- `Attrition_Flag`와 인구통계/카드등급/소득 등 profiling용 변수는 clustering input에서 제외한다.
- 행동 축을 만들기 전에 간단한 검증 지표를 확인한다.
- 통합이 가능한 변수는 상관관계와 의미가 모두 유사한 경우에만 묶는다.
- 이상치 처리는 분포를 확인한 뒤 적용한다.
- 모든 경로는 프로젝트 루트 기준 상대경로를 사용한다.

In [1]:
from pathlib import Path
import sys
import pickle

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for path in [current] + list(current.parents):
        if (path / "data").exists() and (path / "src").exists():
            return path
    return current


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

from src.common import read_csv_safely, save_csv, save_json

RANDOM_STATE = 42
print("Project root:", ROOT)

Project root: C:\Users\user\Desktop\BankChurners-Customer-Segmentation


## 1. 데이터 로드

입력 파일은 팀원이 Step 3까지 완료한 최종 전처리 CSV이다. 이 파일에는 기본 정제, 범주형 인코딩, log transform, 일부 scaling 결과가 포함되어 있다.

In [2]:
input_path = ROOT / "data" / "interim" / "BankChurners_Final_Preprocessed.csv"
output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

if not input_path.exists():
    raise FileNotFoundError(
        f"입력 파일이 없습니다: {input_path}"
        "data/interim/BankChurners_Final_Preprocessed.csv 위치에 파일을 배치하세요."
    )

df = read_csv_safely(input_path)
print("Loaded:", input_path.relative_to(ROOT))
print("Shape:", df.shape)
display(df.head())

Loaded: data\interim\BankChurners_Final_Preprocessed.csv
Shape: (10127, 36)


,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,...,Total_Trans_Amt_log_scaled,Months_Inactive_12_mon_scaled,Total_Ct_Chng_Q4_Q1_scaled,Total_Amt_Chng_Q4_Q1_scaled,Total_Relationship_Count_scaled,Months_on_book_scaled,Contacts_Count_12_mon_scaled,Credit_Limit_log_scaled,Total_Revolving_Bal_log_scaled,Avg_Utilization_Ratio_scaled
0,0,45,1,3,3,1,2,0,39,5,...,-1.715043,-1.327136,3.834003,2.623494,0.763943,0.384621,0.492404,0.905210,0.371202,-0.775882
1,0,49,0,5,2,2,4,0,44,6,...,-1.530414,-1.327136,12.608573,3.563293,1.407306,1.010715,-0.411616,0.444695,0.404962,-0.616276
2,0,51,1,3,2,1,3,0,36,4,...,-0.950593,-1.327136,6.807864,8.367214,0.120579,0.008965,-2.219655,-0.499786,-1.748867,-0.997155
3,0,40,0,4,3,3,4,0,34,3,...,-1.679416,1.641478,6.807864,2.942843,-0.522785,-0.241473,-1.315636,-0.533199,0.745260,1.759686
4,0,40,1,3,5,1,2,0,21,5,...,-2.230961,-1.327136,7.509325,6.455682,0.763943,-1.869317,-2.219655,-0.155051,-1.748867,-0.997155


## 2. 기본 검증

이후 feature engineering에 필요한 핵심 컬럼이 존재하는지 확인한다. 결측치와 중복 row도 함께 확인한다.

In [3]:
required_columns = [
    "Attrition_Flag", "Gender", "Education_Level", "Marital_Status", "Income_Category", "Card_Category",
    "Customer_Age", "Dependent_count", "Months_on_book", "Total_Relationship_Count",
    "Months_Inactive_12_mon", "Contacts_Count_12_mon", "Credit_Limit", "Total_Revolving_Bal",
    "Avg_Open_To_Buy", "Avg_Utilization_Ratio", "Total_Trans_Amt", "Total_Trans_Ct",
    "Total_Amt_Chng_Q4_Q1", "Total_Ct_Chng_Q4_Q1"
]

missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    raise ValueError(f"필수 컬럼 누락: {missing_cols}")

basic_validation = pd.DataFrame({
    "item": ["n_rows", "n_columns", "missing_values", "duplicated_rows"],
    "value": [len(df), df.shape[1], int(df.isnull().sum().sum()), int(df.duplicated().sum())]
})

display(basic_validation)

,item,value
0,n_rows,10127
1,n_columns,36
2,missing_values,0
3,duplicated_rows,0


## 3. Profiling용 label 복원

범주형 변수의 인코딩 값은 거리 계산에서 의미 있는 수치가 아니므로 clustering input에 사용하지 않는다. Segment profiling에서 사람이 해석할 수 있도록 label을 복원한다.

In [4]:
label_maps = {
    "Attrition_Label": ("Attrition_Flag", {0: "Existing Customer", 1: "Attrited Customer"}),
    "Gender_Label": ("Gender", {0: "F", 1: "M"}),
    "Education_Label": ("Education_Level", {
        0: "College", 1: "Doctorate", 2: "Graduate", 3: "High School",
        4: "Post-Graduate", 5: "Uneducated", 6: "Unknown"
    }),
    "Marital_Label": ("Marital_Status", {0: "Divorced", 1: "Married", 2: "Single", 3: "Unknown"}),
    "Income_Label": ("Income_Category", {
        0: "$120K +", 1: "$40K - $60K", 2: "$60K - $80K",
        3: "$80K - $120K", 4: "Less than $40K", 5: "Unknown"
    }),
    "Card_Label": ("Card_Category", {0: "Blue", 1: "Gold", 2: "Platinum", 3: "Silver"})
}

for new_col, (old_col, mapping) in label_maps.items():
    df[new_col] = df[old_col].map(mapping)

label_nulls = df[list(label_maps.keys())].isnull().sum().to_frame("n_missing")
display(label_nulls)

,n_missing
Attrition_Label,0
Gender_Label,0
Education_Label,0
Marital_Label,0
Income_Label,0
Card_Label,0


## 4. Feature 설계 근거 확인

행동 축을 만들기 전에 핵심 변수의 관계를 간단히 확인한다. 이 셀의 목적은 feature를 불필요하게 복잡하게 만드는 것이 아니라, 변수를 통합하거나 제외하는 주요 판단 근거를 남기는 것이다.

In [5]:
# Avg_Open_To_Buy는 가용 한도 성격의 변수이므로 Credit_Limit, Total_Revolving_Bal과의 산술 관계를 확인한다.
open_to_buy_error = df["Credit_Limit"] - df["Total_Revolving_Bal"] - df["Avg_Open_To_Buy"]

# 변수 통합/분리 판단에 필요한 핵심 상관관계를 확인한다.
feature_evidence = pd.DataFrame([
    {
        "check": "open_to_buy_identity_max_abs_error",
        "value": open_to_buy_error.abs().max(),
        "interpretation": "0에 가까우면 Avg_Open_To_Buy가 Credit_Limit - Total_Revolving_Bal과 중복됨"
    },
    {
        "check": "activity_pair_corr",
        "value": df["Total_Trans_Ct"].corr(np.log1p(df["Total_Trans_Amt"])),
        "interpretation": "거래 횟수와 log 거래 금액이 카드 사용 활성도라는 공통 축으로 묶일 수 있는지 확인"
    },
    {
        "check": "decline_pair_corr",
        "value": df["Total_Ct_Chng_Q4_Q1"].corr(df["Total_Amt_Chng_Q4_Q1"]),
        "interpretation": "거래 횟수 변화율과 거래 금액 변화율을 하나로 묶을지 또는 분리할지 확인"
    },
    {
        "check": "relationship_pair_corr",
        "value": df["Total_Relationship_Count"].corr(df["Months_on_book"]),
        "interpretation": "상품 수와 이용 기간을 하나의 관계 강도 축으로 묶을 수 있는지 확인"
    },
    {
        "check": "attrition_corr_total_relationship_count",
        "value": df["Attrition_Flag"].corr(df["Total_Relationship_Count"]),
        "interpretation": "관계 상품 수와 이탈 여부의 선형 관계 참고"
    },
    {
        "check": "attrition_corr_months_on_book",
        "value": df["Attrition_Flag"].corr(df["Months_on_book"]),
        "interpretation": "이용 기간과 이탈 여부의 선형 관계 참고"
    },
    {
        "check": "attrition_corr_count_change",
        "value": df["Attrition_Flag"].corr(df["Total_Ct_Chng_Q4_Q1"]),
        "interpretation": "거래 횟수 변화율의 사후 검증용 이탈 관련성 참고"
    },
    {
        "check": "attrition_corr_amount_change",
        "value": df["Attrition_Flag"].corr(df["Total_Amt_Chng_Q4_Q1"]),
        "interpretation": "거래 금액 변화율의 사후 검증용 이탈 관련성 참고"
    },
])

save_csv(feature_evidence, output_dir / "feature_design_evidence.csv")
display(feature_evidence)

,check,value,interpretation
0,open_to_buy_identity_max_abs_error,5.684342e-14,0에 가까우면 Avg_Open_To_Buy가 Credit_Limit - Total_...
1,activity_pair_corr,9.069571e-01,거래 횟수와 log 거래 금액이 카드 사용 활성도라는 공통 축으로 묶일 수 있는지 확인
2,decline_pair_corr,3.841893e-01,거래 횟수 변화율과 거래 금액 변화율을 하나로 묶을지 또는 분리할지 확인
3,relationship_pair_corr,-9.203080e-03,상품 수와 이용 기간을 하나의 관계 강도 축으로 묶을 수 있는지 확인
4,attrition_corr_total_relationship_count,-1.500052e-01,관계 상품 수와 이탈 여부의 선형 관계 참고
5,attrition_corr_months_on_book,1.368685e-02,이용 기간과 이탈 여부의 선형 관계 참고
6,attrition_corr_count_change,-2.900540e-01,거래 횟수 변화율의 사후 검증용 이탈 관련성 참고
7,attrition_corr_amount_change,-1.310628e-01,거래 금액 변화율의 사후 검증용 이탈 관련성 참고


## 5. 변화율 변수 이상치 확인 및 clipping

거래 변화율 변수는 일부 극단값이 거리 기반 clustering에 영향을 줄 수 있다. 분위수와 IQR 기준을 확인한 뒤, 1%~99% 분위수 범위로 clipping한다.

In [6]:
def summarize_outliers(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        s = frame[col]
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr
        rows.append({
            "feature": col,
            "min": s.min(), "q01": s.quantile(0.01), "median": s.median(),
            "q99": s.quantile(0.99), "max": s.max(),
            "iqr_lower_fence": lower_fence, "iqr_upper_fence": upper_fence,
            "n_below_fence": int((s < lower_fence).sum()),
            "n_above_fence": int((s > upper_fence).sum()),
        })
    return pd.DataFrame(rows)

change_cols = ["Total_Ct_Chng_Q4_Q1", "Total_Amt_Chng_Q4_Q1"]
outlier_summary = summarize_outliers(df, change_cols)
save_csv(outlier_summary, output_dir / "change_rate_outlier_summary.csv")
display(outlier_summary)

clip_thresholds = {}
for col in change_cols:
    lo, hi = df[col].quantile([0.01, 0.99])
    clip_thresholds[col] = {"lower_q01": float(lo), "upper_q99": float(hi)}
    df[f"{col}_clip"] = df[col].clip(lo, hi)

save_json(clip_thresholds, output_dir / "clipping_thresholds.json")

,feature,min,q01,median,q99,max,iqr_lower_fence,iqr_upper_fence,n_below_fence,n_above_fence
0,Total_Ct_Chng_Q4_Q1,0.0,0.23100,0.702,1.45500,3.714,0.228,1.172,96,298
1,Total_Amt_Chng_Q4_Q1,0.0,0.34026,0.736,1.51474,3.397,0.289,1.201,48,348


## 6. Profiling용 파생변수 생성

아래 변수들은 clustering input이 아니라 segment가 만들어진 뒤 고객군의 의미를 해석하기 위한 profiling 변수이다.

In [7]:
df["Avg_Ticket_Size"] = np.where(
    df["Total_Trans_Ct"] > 0,
    df["Total_Trans_Amt"] / df["Total_Trans_Ct"],
    0
)
df["Avg_Ticket_Size_log"] = np.log1p(df["Avg_Ticket_Size"])
df["Inactive_Ratio_12m"] = df["Months_Inactive_12_mon"] / 12
df["Contact_Rate_12m"] = df["Contacts_Count_12_mon"] / 12

df["No_Revolving_Bal_Flag"] = (df["Total_Revolving_Bal"] == 0).astype(int)
df["Low_Bank_Product_Count_Flag"] = (df["Total_Relationship_Count"] <= 2).astype(int)
df["Zero_Utilization_Flag"] = (df["Avg_Utilization_Ratio"] == 0).astype(int)

profiling_cols = [
    "Avg_Ticket_Size", "Avg_Ticket_Size_log", "Inactive_Ratio_12m", "Contact_Rate_12m",
    "No_Revolving_Bal_Flag", "Low_Bank_Product_Count_Flag", "Zero_Utilization_Flag"
]
display(df[profiling_cols].head())

,Avg_Ticket_Size,Avg_Ticket_Size_log,Inactive_Ratio_12m,Contact_Rate_12m,No_Revolving_Bal_Flag,Low_Bank_Product_Count_Flag,Zero_Utilization_Flag
0,27.238095,3.340672,0.083333,0.250000,0,0,0
1,39.121212,3.691905,0.083333,0.166667,0,0,0
2,94.350000,4.557554,0.083333,0.000000,1,0,1
3,58.550000,4.086816,0.333333,0.083333,0,0,0
4,29.142857,3.405948,0.083333,0.000000,1,0,1


## 7. High-risk signal 정의

High-risk signal은 cluster를 만든 뒤 각 segment가 고위험 행동 신호를 얼마나 포함하는지 확인하기 위한 사후 검증용 변수이다. Clustering input에는 포함하지 않는다.

거래 변화율은 단독으로 위험을 단정하기보다 낮은 활동성 또는 거래 금액 감소와 결합하여 사용한다.

In [8]:
q = {
    "ct_change_q20": df["Total_Ct_Chng_Q4_Q1"].quantile(0.20),
    "amt_change_q20": df["Total_Amt_Chng_Q4_Q1"].quantile(0.20),
    "trans_ct_q20": df["Total_Trans_Ct"].quantile(0.20),
}

low_ct_change = df["Total_Ct_Chng_Q4_Q1"] <= q["ct_change_q20"]
low_amt_change = df["Total_Amt_Chng_Q4_Q1"] <= q["amt_change_q20"]
low_activity_ct = df["Total_Trans_Ct"] <= q["trans_ct_q20"]

df["Low_Ct_Change_Flag"] = low_ct_change.astype(int)
df["Low_Amt_Change_Flag"] = low_amt_change.astype(int)
df["Strong_Decline_Flag"] = (low_ct_change & low_amt_change).astype(int)
df["Decline_Low_Activity_Flag"] = (low_ct_change & low_activity_ct).astype(int)

df["High_Inactive_Flag"] = (df["Months_Inactive_12_mon"] >= 4).astype(int)
df["High_Contact_Flag"] = (df["Contacts_Count_12_mon"] >= 4).astype(int)

risk_signal_cols = [
    "Strong_Decline_Flag",
    "Decline_Low_Activity_Flag",
    "High_Inactive_Flag",
    "High_Contact_Flag",
    "Low_Bank_Product_Count_Flag",
    "Zero_Utilization_Flag",
]

df["High_Risk_Signal_Count"] = df[risk_signal_cols].sum(axis=1)

risk_signal_summary = pd.DataFrame({
    "feature": risk_signal_cols,
    "flag_ratio": [df[col].mean() for col in risk_signal_cols],
    "attrition_rate_when_flagged": [df.loc[df[col] == 1, "Attrition_Flag"].mean() for col in risk_signal_cols],
})

save_csv(risk_signal_summary, output_dir / "risk_signal_summary.csv")
save_json({**q, "high_inactive_threshold": 4, "high_contact_threshold": 4}, output_dir / "risk_signal_thresholds.json")
display(risk_signal_summary)

,feature,flag_ratio,attrition_rate_when_flagged
0,Strong_Decline_Flag,0.079787,0.467822
1,Decline_Low_Activity_Flag,0.090056,0.527412
2,High_Inactive_Flag,0.072776,0.245590
3,High_Contact_Flag,0.160166,0.263872
4,Low_Bank_Product_Count_Flag,0.212600,0.268927
5,Zero_Utilization_Flag,0.243902,0.361538


## 8. 관계 상품 수 기준 확인

`Total_Relationship_Count`는 구매 물품 수가 아니라 고객이 은행에서 보유한 금융 상품 수이다. 관계 상품 수가 낮은 고객군을 정의하기 위해 상품 수별 이탈률을 확인한다.

In [9]:
relationship_count_summary = df.groupby("Total_Relationship_Count").agg(
    n_customers=("Attrition_Flag", "size"),
    attrition_rate=("Attrition_Flag", "mean"),
    mean_total_trans_ct=("Total_Trans_Ct", "mean"),
    mean_total_trans_amt=("Total_Trans_Amt", "mean"),
).reset_index()
relationship_count_summary["customer_ratio"] = relationship_count_summary["n_customers"] / len(df)

save_csv(relationship_count_summary, output_dir / "relationship_count_evidence.csv")
display(relationship_count_summary)

,Total_Relationship_Count,n_customers,attrition_rate,mean_total_trans_ct,mean_total_trans_amt,customer_ratio
0,1,910,0.256044,79.619780,7252.917582,0.089859
1,2,1243,0.278359,77.081255,7019.148029,0.122741
2,3,2305,0.173536,62.105857,3936.279393,0.227609
3,4,1912,0.117678,61.753661,3754.970711,0.188802
4,5,1891,0.120042,60.810153,3515.332628,0.186729
5,6,1866,0.105038,60.203108,3416.454448,0.184260


## 9. 행동 축 생성

행동 축은 다음 원칙으로 구성한다.

- 의미가 유사하고 상관관계가 높은 변수는 하나의 축으로 통합한다.
- 의미가 유사하더라도 상관관계가 충분히 높지 않으면 분리한다.
- 이탈 여부는 행동 축 생성에 사용하지 않고, 사후 profiling에서만 사용한다.
- 모든 component는 먼저 표준화하고, 생성된 행동 축도 한 번 더 표준화하여 clustering에서 축별 영향력을 맞춘다.

In [10]:
component_features = pd.DataFrame({
    "Total_Trans_Ct": df["Total_Trans_Ct"],
    "Total_Trans_Amt_log": np.log1p(df["Total_Trans_Amt"]),
    "Total_Ct_Chng_Q4_Q1_clip": df["Total_Ct_Chng_Q4_Q1_clip"],
    "Total_Amt_Chng_Q4_Q1_clip": df["Total_Amt_Chng_Q4_Q1_clip"],
    "Months_Inactive_12_mon": df["Months_Inactive_12_mon"],
    "Total_Relationship_Count": df["Total_Relationship_Count"],
    "Contacts_Count_12_mon": df["Contacts_Count_12_mon"],
    "Avg_Utilization_Ratio": df["Avg_Utilization_Ratio"],
    "Credit_Limit_log": np.log1p(df["Credit_Limit"]),
}, index=df.index)

component_scaler = StandardScaler()
component_z = pd.DataFrame(
    component_scaler.fit_transform(component_features),
    columns=component_features.columns,
    index=df.index,
)

axis_features_raw = pd.DataFrame(index=df.index)
axis_features_raw["Activity_Score"] = component_z[["Total_Trans_Ct", "Total_Trans_Amt_log"]].mean(axis=1)
axis_features_raw["Count_Decline_Risk_Score"] = -component_z["Total_Ct_Chng_Q4_Q1_clip"]
axis_features_raw["Amount_Decline_Risk_Score"] = -component_z["Total_Amt_Chng_Q4_Q1_clip"]
axis_features_raw["Inactivity_Score"] = component_z["Months_Inactive_12_mon"]
axis_features_raw["Product_Relationship_Score"] = component_z["Total_Relationship_Count"]
axis_features_raw["Contact_Burden_Score"] = component_z["Contacts_Count_12_mon"]
axis_features_raw["Credit_Utilization_Score"] = component_z["Avg_Utilization_Ratio"]
axis_features_raw["Credit_Capacity_Score"] = component_z["Credit_Limit_log"]

axis_scaler = StandardScaler()
df_cluster_ready = pd.DataFrame(
    axis_scaler.fit_transform(axis_features_raw),
    columns=axis_features_raw.columns,
    index=df.index,
)

df_profile_ready = df.copy()

print("component_features:", component_features.shape)
print("axis_features_raw:", axis_features_raw.shape)
print("df_cluster_ready:", df_cluster_ready.shape)
display(df_cluster_ready.head())

component_features: (10127, 9)
axis_features_raw: (10127, 8)
df_cluster_ready: (10127, 8)


,Activity_Score,Count_Decline_Risk_Score,Amount_Decline_Risk_Score,Inactivity_Score,Product_Relationship_Score,Contact_Burden_Score,Credit_Utilization_Score,Credit_Capacity_Score
0,-1.376877,-3.531338,-2.853554,-1.327136,0.763943,0.492404,-0.775882,0.905210
1,-1.478682,-3.531338,-3.742487,-1.327136,1.407306,-0.411616,-0.616276,0.444695
2,-1.465391,-3.531338,-3.742487,-1.327136,0.120579,-2.219655,-0.997155,-0.499786
3,-1.838587,-3.531338,-3.199750,1.641478,-0.522785,-1.315636,1.759686,-0.533199
4,-1.946479,-3.531338,-3.742487,-1.327136,0.763943,-2.219655,-0.997155,-0.155051


## 10. 최종 파일 저장

`BankChurners_cluster_input.csv`는 Step 4 clustering에 사용되는 행동 축 데이터이다. `BankChurners_profile_table.csv`는 Step 5 segment profiling에 사용되는 해석용 테이블이다.

In [11]:
# 저장 전 검증
forbidden_cols = [
    "Attrition_Flag", "Attrition_Label", "Gender", "Education_Level", "Marital_Status",
    "Income_Category", "Card_Category", "Customer_Age", "Dependent_count"
]
included_forbidden = [col for col in forbidden_cols if col in df_cluster_ready.columns]
if included_forbidden:
    raise ValueError(f"clustering input에 포함되면 안 되는 컬럼이 있습니다: {included_forbidden}")

if df_cluster_ready.isnull().sum().sum() > 0:
    raise ValueError("df_cluster_ready에 결측치가 있습니다.")

# Row_ID 부여
cluster_save = df_cluster_ready.reset_index(drop=True).copy()
profile_save = df_profile_ready.reset_index(drop=True).copy()
axis_save = axis_features_raw.reset_index(drop=True).copy()
component_save = component_features.reset_index(drop=True).copy()

for frame in [cluster_save, profile_save, axis_save, component_save]:
    frame.insert(0, "Row_ID", np.arange(len(frame)))

save_csv(cluster_save, output_dir / "BankChurners_cluster_input.csv")
save_csv(profile_save, output_dir / "BankChurners_profile_table.csv")
save_csv(axis_save, output_dir / "BankChurners_axis_features_raw.csv")
save_csv(component_save, output_dir / "BankChurners_component_features.csv")

feature_sets = {
    "core_7": [
        "Activity_Score", "Count_Decline_Risk_Score", "Amount_Decline_Risk_Score",
        "Inactivity_Score", "Product_Relationship_Score", "Contact_Burden_Score",
        "Credit_Utilization_Score",
    ],
    "core_7_plus_capacity": [
        "Activity_Score", "Count_Decline_Risk_Score", "Amount_Decline_Risk_Score",
        "Inactivity_Score", "Product_Relationship_Score", "Contact_Burden_Score",
        "Credit_Utilization_Score", "Credit_Capacity_Score",
    ]
}

feature_info = {
    "step": "Step 3.5 Further Preprocessing",
    "input_file": str(input_path.relative_to(ROOT)),
    "output_files": {
        "cluster_input": "data/processed/BankChurners_cluster_input.csv",
        "profile_table": "data/processed/BankChurners_profile_table.csv",
        "axis_features_raw": "data/processed/BankChurners_axis_features_raw.csv",
        "component_features": "data/processed/BankChurners_component_features.csv",
    },
    "feature_sets": feature_sets,
    "cluster_input_columns": list(df_cluster_ready.columns),
    "risk_signal_columns": risk_signal_cols,
    "notes": [
        "Attrition_Flag is excluded from clustering input and used only for profiling.",
        "Demographic and label-encoded categorical variables are excluded from clustering input.",
        "Credit_Capacity_Score is available for comparison but not required in the core behavior feature set.",
    ]
}

save_json(feature_info, output_dir / "cluster_feature_info.json")

with open(output_dir / "preprocessing_scalers.pkl", "wb") as f:
    pickle.dump({"component_scaler": component_scaler, "axis_scaler": axis_scaler}, f)

validation_summary = pd.DataFrame({
    "item": ["n_rows", "n_cluster_columns", "n_profile_columns", "cluster_missing_values", "profile_missing_values"],
    "value": [
        len(cluster_save), len(df_cluster_ready.columns), len(profile_save.columns),
        int(cluster_save.isnull().sum().sum()), int(profile_save.isnull().sum().sum())
    ]
})
save_csv(validation_summary, output_dir / "preprocessing_validation_summary.csv")

display(validation_summary)
print("Step 3.5 completed. Files saved to:", output_dir.relative_to(ROOT))

,item,value
0,n_rows,10127
1,n_cluster_columns,8
2,n_profile_columns,59
3,cluster_missing_values,0
4,profile_missing_values,0


Step 3.5 completed. Files saved to: data\processed
